In [11]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# 1. Setup time vector
t_max = 2 * np.pi
dt = 0.005 # Ultra-fine resolution for peak stability
t = np.arange(0, t_max, dt)

# 2. Define target path (Lemniscate of Bernoulli)
a = 10
x_ref = (a * np.cos(t)) / (1 + np.sin(t)**2)
y_ref = (a * np.sin(t) * np.cos(t)) / (1 + np.sin(t)**2)

# 3. Simulation Parameters & Disturbance
wind_x, wind_y = 1.5, 1.0
kp = 150.0 # Maximum responsiveness
ki = 50.0  # Aggressive error elimination
kd = 20.0  # High damping for stability

# Initial state: Start exactly on path
x, y = x_ref[0], y_ref[0]
vx, vy = 0.0, 0.0

x_history, y_history = [], []
prev_ex, prev_ey = 0.0, 0.0
integral_x, integral_y = 0.0, 0.0

# 4. Implement PID Control Loop
for i in range(len(t)):
    target_x, target_y = x_ref[i], y_ref[i]
    ex, ey = target_x - x, target_y - y

    # Integral accumulation
    integral_x += ex * dt
    integral_y += ey * dt

    # Derivative calculation
    dex = (ex - prev_ex) / dt
    dey = (ey - prev_ey) / dt

    # PID Control Signal (Acceleration)
    ax_sig = kp * ex + ki * integral_x + kd * dex
    ay_sig = kp * ey + ki * integral_y + kd * dey

    # Physics update with wind disturbance
    vx += (ax_sig + wind_x) * dt
    vy += (ay_sig + wind_y) * dt
    x += vx * dt
    y += vy * dt

    x_history.append(x)
    y_history.append(y)
    prev_ex, prev_ey = ex, ey

# 5. Create Dynamic Animation
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(-12, 12)
ax.set_ylim(-6, 6)
ax.set_aspect('equal')
ax.grid(True)
ax.set_title("Ultra-Precision Drone Tracking: Final PID Tuning")

ax.plot(x_ref, y_ref, 'k--', lw=1.5, alpha=0.4, label='Target Path')
line, = ax.plot([], [], 'b-', lw=2, label='Drone Trajectory (PID)')
drone_dot, = ax.plot([], [], 'ro', markersize=7, label='Drone')
ax.legend(loc='upper right')

def init():
    line.set_data([], [])
    drone_dot.set_data([], [])
    return line, drone_dot

def animate(i):
    # Subsample (10x) to maintain reasonable playback speed
    idx = min(i * 10, len(x_history) - 1)
    line.set_data(x_history[:idx], y_history[:idx])
    drone_dot.set_data([x_history[idx]], [y_history[idx]])
    return line, drone_dot

anim = animation.FuncAnimation(fig, animate, init_func=init, frames=len(t)//10, interval=25, blit=True)
plt.close()

try:
    display_obj = HTML(anim.to_html5_video())
except Exception:
    display_obj = HTML(anim.to_jshtml())

display_obj